In [7]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",1,1,0,NaN,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.0000,0.122,0.535,157.969,3,Lower
1,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",2,1,4,NaN,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.0000,0.248,0.576,138.008,4,About_Average
2,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,3,-2,8,NaN,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.0000,0.141,0.214,101.061,4,Higher
3,4wJ5Qq0jBN4ajy7ouZIV1c,APT.,"ROSÉ, Bruno Mars",4,0,-2,NaN,2025-02-17,89,False,...,-4.477,0,0.2600,0.0283,0.0000,0.355,0.939,149.027,4,Higher
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,-2,NaN,2025-02-17,96,False,...,-10.171,1,0.0358,0.2000,0.0608,0.117,0.438,104.978,4,About_Average


In [8]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # or tensorflow, or torch

import keras
from keras import layers, ops

from sklearn.model_selection import train_test_split

from ast import literal_eval
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# test_split = 0.2

# # Initial train and test split.
# train_df, test_df = train_test_split(
#     df,
#     test_size=test_split,
#     stratify=df["average_song"].values,
# )

# # Splitting the test set further into validation
# # and new test sets.
# val_df = test_df.sample(frac=0.5)
# test_df.drop(val_df.index, inplace=True)

# print(f"Number of rows in training set: {len(train_df)}")
# print(f"Number of rows in validation set: {len(val_df)}")
# print(f"Number of rows in test set: {len(test_df)}")


Number of rows in training set: 1382705
Number of rows in validation set: 172838
Number of rows in test set: 172839


In [ ]:
# # For RaggedTensor
# import tensorflow as tf

# terms = tf.ragged.constant(train_df["average_song"].values)
# lookup = layers.StringLookup(output_mode="multi_hot")
# lookup.adapt(terms)
# vocab = lookup.get_vocabulary()


# def invert_multi_hot(encoded_labels):
#     """Reverse a single multi-hot encoded label to a tuple of vocab terms."""
#     hot_indices = np.argwhere(encoded_labels == 1.0)[..., 0]
#     return np.take(vocab, hot_indices)


# print("Vocabulary:\n")
# print(vocab)

# sample_label = train_df["average_song"].iloc[0]
# print(f"Original label: {sample_label}")

# label_binarized = lookup([sample_label])
# print(f"Label-binarized representation: {label_binarized}")


Vocabulary:

['[UNK]', np.str_('Higher'), np.str_('About_Average'), np.str_('Lower')]
Original label: Lower
Label-binarized representation: [0 0 0 1]


In [ ]:
# COLUMN_NAMES = [
#     "age",
#     "sex",
#     "cp",
#     "trestbps",
#     "chol",
#     "fbs",
#     "restecg",
#     "thalach",
#     "exang",
#     "oldpeak",
#     "slope",
#     "ca",
#     "thal",
#     "target",
# ]
# # Target feature name.
# TARGET_FEATURE_NAME = "average_song"
# # Numeric feature names.
# NUMERIC_FEATURE_NAMES = ["key", "mode", "liveness", "speechiness", "loudness", "loudness", "tempo", "duration_ms", "time_signature", "acousticness", "instrumentalness", "valence"]
# # Categorical features and their vocabulary lists.
# # Note that we add 'v=' as a prefix to all categorical feature values to make
# # sure that they are treated as strings.

# CATEGORICAL_FEATURES_WITH_VOCABULARY = {
#     feature_name: sorted(
#         [
#             # Integer categorcal must be int and string must be str
#             value if df[feature_name].dtype == "int64" else str(value)
#             for value in list(df[feature_name].unique())
#         ]
#     )
#     for feature_name in COLUMN_NAMES
#     if feature_name not in list(NUMERIC_FEATURE_NAMES + [TARGET_FEATURE_NAME])
# }
# # All features names.
# FEATURE_NAMES = NUMERIC_FEATURE_NAMES + list(
#     CATEGORICAL_FEATURES_WITH_VOCABULARY.keys()
# )

# def dataframe_to_dataset(dataframe):
#     dataframe = dataframe.copy()
#     labels = dataframe.pop("target")
#     ds = tf.data.Dataset.from_tensor_slices((dict(dataframe), labels)).map(
#         encode_categorical
#     )
#     ds = ds.shuffle(buffer_size=len(dataframe))
#     return ds

# # We process our datasets elements here (categorical) and convert them to indices to avoid this step
# # during model training since only tensorflow support strings.
# def encode_categorical(features, target):
#     for feature_name in features:
#         if feature_name in CATEGORICAL_FEATURES_WITH_VOCABULARY:
#             lookup_class = (
#                 layers.StringLookup
#                 if features[feature_name].dtype == "string"
#                 else layers.IntegerLookup
#             )
#             vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]
#             # Create a lookup to convert a string values to an integer indices.
#             # Since we are not using a mask token nor expecting any out of vocabulary
#             # (oov) token, we set mask_token to None and  num_oov_indices to 0.
#             index = lookup_class(
#                 vocabulary=vocabulary,
#                 mask_token=None,
#                 num_oov_indices=0,
#                 output_mode="binary",
#             )
#             # Convert the string input values into integer indices.
#             value_index = index(features[feature_name])
#             features[feature_name] = value_index

#         else:
#             pass

#     # Change features from OrderedDict to Dict to match Inputs as they are Dict.
#     return dict(features), target


# def encode_numerical_feature(feature, name, dataset):
#     # Create a Normalization layer for our feature
#     normalizer = layers.Normalization()
#     # Prepare a Dataset that only yields our feature
#     feature_ds = dataset.map(lambda x, y: x[name])
#     feature_ds = feature_ds.map(lambda x: tf.expand_dims(x, -1))
#     # Learn the statistics of the data
#     normalizer.adapt(feature_ds)
#     # Normalize the input feature
#     encoded_feature = normalizer(feature)
#     return encoded_feature


# train_ds = dataframe_to_dataset(train_dataframe)
# val_ds = dataframe_to_dataset(val_dataframe)


In [24]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = df.drop(columns=["average_song", "spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date"], axis=1, inplace=False)
y = df["average_song"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()



In [25]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp_pipeline = make_pipeline(scaler, mlp)
mlp_pipeline.fit(X_train, y_train)
y_pred = mlp_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
cm

               precision    recall  f1-score   support

About_Average       0.99      0.99      0.99    114033
       Higher       0.99      0.99      0.99    130484
        Lower       0.99      0.99      0.99    101160

     accuracy                           0.99    345677
    macro avg       0.99      0.99      0.99    345677
 weighted avg       0.99      0.99      0.99    345677



array([[113459,    464,    110],
       [   415, 129640,    429],
       [   192,    659, 100309]])